In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

## Load Results

In [ ]:
seeds = [42, 67, 99, 70, 73]

datasets = {
    "EchoNext": {
        72475: "runs-echonext",
        32768: "runs-echonext-32k",
        16384: "runs-echonext-16k",
        8192: "runs-echonext-8k",
        4096: "runs-echonext-4k",
        2048: "runs-echonext-2k",
        1024: "runs-echonext-1k",
        512: "runs-echonext-512",
        256: "runs-echonext-256",
    },
    "MIMIC-IV-ECG": {
        78470: "runs-mimic",
        32768: "runs-mimic-32k",
        16384: "runs-mimic-16k",
        8192: "runs-mimic-8k",
        4096: "runs-mimic-4k",
        2048: "runs-mimic-2k",
        1024: "runs-mimic-1k",
        512: "runs-mimic-512",
        256: "runs-mimic-256",
    },
    "CODE-15%": {
        74112: "runs-code15",
        32768: "runs-code15-32k",
        16384: "runs-code15-16k",
        8192: "runs-code15-8k",
        4096: "runs-code15-4k",
        2048: "runs-code15-2k",
        1024: "runs-code15-1k",
        512: "runs-code15-512",
        256: "runs-code15-256",
    },
    "PTB-XL": {
        17418: "runs-ptbxl",
        8722: "runs-ptbxl-8k",
        4356: "runs-ptbxl-4k",
        2175: "runs-ptbxl-2k",
        1091: "runs-ptbxl-1k",
        547: "runs-ptbxl-512",
        273: "runs-ptbxl-256",
    },
    "CinC Georgia": {
        8192: "runs-cinc",
        4096: "runs-cinc-4k",
        2048: "runs-cinc-2k",
        1024: "runs-cinc-1k",
        512: "runs-cinc-512",
        256: "runs-cinc-256",
    },
    "ZZU pECG": {
        8658: "runs-zzu",
        4096: "runs-zzu-4k",
        2048: "runs-zzu-2k",
        1024: "runs-zzu-1k",
        512: "runs-zzu-512",
        256: "runs-zzu-256",
    },
}

# mapping of experiment name to tuple of:
# - plotting color
# - folder name
experiments = {
    "SupProto Direct":     ("tab:green",  "labsup-proto-direct"),
    "SupProto HEEDB":      ("tab:orange", "labsup-proto-heedb-rila"),
    "SupProto HEEDB (FT)": ("tab:pink",   "labsup-proto-heedb-rila-ft"),
    "ProtoSSL HEEDB":      ("tab:blue",   "protossl-heedb-pila"),
    "ProtoSSL HEEDB (FT)": ("tab:cyan",   "protossl-heedb-pila-ft"),
    ###
    "Blackbox Direct":     ("tab:grey",   "blackbox-direct"),
    "ECGFounder":          ("tab:red",    "ecgfounder-logreg"),
    "ST-MEM":              ("tab:purple", "stmem-logreg"),
    ###
    "SupProto HEEDB (PILA)":      ("tab:red", "labsup-proto-heedb-pila"),
    "SupProto HEEDB (PILA) (FT)": ("tab:grey",   "labsup-proto-heedb-pila-ft"),
    ###
    "SupProto Direct (7ppl)":     ("tab:green",  "labsup-proto-direct-7ppl"),
    "SupProto HEEDB (7ppl)":      ("tab:orange", "labsup-proto-heedb-rila-7ppl"),
    "SupProto HEEDB (7ppl) (FT)": ("tab:pink",   "labsup-proto-heedb-rila-7ppl-ft"),
    "ProtoSSL HEEDB (7ppl)":      ("tab:blue",     "protossl-heedb-pila-7ppl"),
    "ProtoSSL HEEDB (7ppl) (FT)": ("tab:cyan",   "protossl-heedb-pila-7ppl-ft"),
    ###
    "SupProto Direct (28ppl)":     ("tab:green",  "labsup-proto-direct-28ppl"),
    "SupProto HEEDB (28ppl)":      ("tab:orange", "labsup-proto-heedb-rila-28ppl"),
    "SupProto HEEDB (28ppl) (FT)": ("tab:pink",   "labsup-proto-heedb-rila-28ppl-ft"),
    "ProtoSSL HEEDB (28ppl)":      ("tab:blue",  "protossl-heedb-pila-28ppl"),
    "ProtoSSL HEEDB (28ppl) (FT)": ("tab:cyan",  "protossl-heedb-pila-28ppl-ft"),
    ###
    "ProtoSSL HEEDB (PIA)": ("cornflowerblue",  "protossl-heedb-pia"),
    "ProtoSSL HEEDB (PIA) (FT)": ("lightsteelblue",  "protossl-heedb-pia-ft"),
    ###
    "ProtoSSL HEEDB (83ppl)": ("yellow", "protossl-heedb-pila-83ppl"),
    "ProtoSSL HEEDB (PIA) (83ppl)": ("lime", "protossl-heedb-pia-83ppl"),
    "ProtoSSL HEEDB (PIT)": ("blue",  "protossl-heedb-pit"),
    "ProtoSSL HEEDB (PIP)": ("red",  "protossl-heedb-pip"),
    ###
    "ECGFounder (LAP)":   ("indianred",  "ecgfounder-lap-1000-init"),
    "ECGFounder (SK-OT)": ("firebrick",  "ecgfounder-clustering"),
    "ECGFounder (Rand)":  ("maroon",  "ecgfounder-random-1000-init"),
}

use_new = [
    "Blackbox Direct",
    "SupProto Direct",
    "SupProto HEEDB",
    "SupProto HEEDB (FT)",
    "SupProto HEEDB (PILA)",
    "SupProto HEEDB (PILA) (FT)",
    "SupProto Direct (7ppl)",
    "SupProto HEEDB (7ppl)",
    "SupProto HEEDB (7ppl) (FT)",
    "SupProto Direct (28ppl)",
    "SupProto HEEDB (28ppl)",
    "SupProto HEEDB (28ppl) (FT)",
    "ProtoSSL HEEDB",
    "ProtoSSL HEEDB (FT)",
    "ProtoSSL HEEDB (PIA)",
    "ProtoSSL HEEDB (PIA) (FT)", # new
    "ProtoSSL HEEDB (PIT)",
    "ProtoSSL HEEDB (PIP)",
    "ProtoSSL HEEDB (7ppl)",
    "ProtoSSL HEEDB (7ppl) (FT)",
    "ProtoSSL HEEDB (28ppl)",
    "ProtoSSL HEEDB (28ppl) (FT)",
    "ProtoSSL HEEDB (83ppl)",
    "ProtoSSL HEEDB (PIA) (83ppl)",
]

def get_palette(exp_names):
    palette = dict()
    exps = experiments
    for exp_name in exp_names:
        palette[exp_name] = exps[exp_name][0]
    return palette

In [ ]:
data = []
for seed in seeds:
    output_dir = Path(f"/opt/gpu_working/steven/protossl-outputs-seed{seed}")
    for ds, sizes in datasets.items():
        for size, run_dir in sizes.items():
            exps = experiments.copy()
            for exp_name, (exp_color, exp_dir) in exps.items():
                if exp_name in use_new:
                    _output_dir = Path(f"/opt/gpu_working/steven/new/protossl-outputs-seed{seed}")
                else:
                    _output_dir = output_dir
                metrics_csv = _output_dir / run_dir / exp_dir / "metrics-bootstrapped-v2.csv"
                if not os.path.exists(metrics_csv):
                    continue
                metrics = pd.read_csv(metrics_csv, index_col="Label")
                multilabel = metrics.loc["Multilabel Averaged"]
                datum = {
                    "Seed": seed,
                    "Dataset": ds,
                    "Model": exp_name,
                    "Train Size": size,
                    "Multilabel (AUROC)": multilabel["AUROC"],
                    "Multilabel (AUPRC)": multilabel["AUPRC"],
                }
                if "AUROC 95% CI (lo)" in metrics.columns:
                    datum["AUROC 95% CI (lo)"] = metrics.loc["Multilabel Averaged", "AUROC 95% CI (lo)"]
                    datum["AUROC 95% CI (hi)"] = metrics.loc["Multilabel Averaged", "AUROC 95% CI (hi)"]
                data.append(datum)
results = pd.DataFrame.from_records(data)

## Plot Label Efficiency Curve

In [ ]:
def plot_lift(
    *,  # enforce kwargs
    df: pd.DataFrame, # long format df (each point to plot is a row)
    dataset: str,
    seed: int | list[int] = 42, # if list of int, average across seeds
    metric: str,
    models: list[str], # must be intentional about which models to plot
    rename: list[str] | None = None,
    baseline_model: str | None = None, # singular result to optionally plot as dashed line
    ylim: tuple[float, float] | None = None,
    xlim: tuple[float, float] | None = None,
    save_path: str | None = None,
    smooth: bool = True,
    ax: plt.Axes | None = None,
    do_label: bool = True,
):
    if rename is not None:
        assert len(models) == len(rename)
    if baseline_model is not None and baseline_model not in models:
        models = [baseline_model] + models
        if rename is not None:
            rename = [baseline_model] + rename
    palette = get_palette(models)
    if rename is not None:
        palette = {new_name: palette[m] for new_name, m in zip(rename, models)}
        df = df.copy()
        df["Model"] = df["Model"].replace({v: k for k, v in zip(rename, models)})

    df = df[df["Dataset"] == dataset]
    if isinstance(seed, int):
        df = df[df["Seed"] == seed]
    else: # list of seeds
        df = df[df["Seed"].isin(seed)]
        df = df.groupby(["Dataset", "Model", "Train Size"]).mean().reset_index()
    # print(dataset, df[metric].min(), df[metric].max())
    fig = None
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 6))
    min_size = df["Train Size"].min()
    max_size = df["Train Size"].max()
    if xlim is not None:
        min_size = min(min_size, xlim[0])
        max_size = max(max_size, xlim[1])

    if baseline_model is not None:
        mask = df["Model"] == baseline_model
        assert (
            mask.sum() == 1
        ), f"Should only have 1 entry for baseline model: {baseline_model}"
        baseline_row = df[mask].iloc[0]
        ax.hlines(
            baseline_row[metric],
            min_size,
            max_size,
            colors=palette.pop(baseline_model),
            linestyles=":",
            label=baseline_model,
        )
        df = df[~mask] # subsequent line plots should exclude baseline model

    if not smooth:
        sns.lineplot(
            df,
            x="Train Size",
            y=metric,
            hue="Model",
            palette=palette,
            hue_order=list(palette.keys()),
            marker="o",
            ax=ax,
        )
    else:
        for model, color in palette.items():
            model_data = df[df["Model"] == model]
            ax.set_xlim([2**7, 2**17])
            sns.regplot(
                model_data,
                x="Train Size",
                y=metric,
                color=color,
                marker="o",
                ax=ax,
                logx=True,
                label=model if do_label else None,
                line_kws={"zorder": 2},
                scatter_kws={"zorder": 3},
                truncate=False,
            )
    ax.set_xscale("log", base=2)
    if xlim is not None:
        ax.set_xlim(xlim)
    else:
        # tighter boundaries than default lims
        ax.set_xlim((min_size, max_size))
    ax.set_title(dataset)
    # ax.set_title(f"{dataset} {metric}")
    if do_label:
        ax.legend(loc="lower right")
    if ylim is not None:
        ax.set_ylim(ylim)
    else:
        ymin, ymax = ax.get_ylim()
        ax.set_ylim((ymin, min(ymax, 1)))
    if save_path is not None:
        assert fig is not None
        fig.tight_layout()
        fig.savefig(save_path)

In [ ]:
import string
from matplotlib.lines import Line2D

def make_paper_fig(models):

    # fig, axs = plt.subplots(nrows=2, ncols=3, figsize=(5.4*2, 3.35*2))
    fig, axs = plt.subplots(nrows=2, ncols=3, figsize=(5.4*2, 2.75*2))
    for i, (ds, short) in enumerate([
        ("EchoNext", "echonext"),
        ("MIMIC-IV-ECG", "mimic"),
        ("ZZU pECG", "zzu"),
        ("PTB-XL", "ptbxl"),
        ("CinC Georgia", "cinc"),
        ("CODE-15%", "code15"),
    ]):
        ax = axs[i//3, i%3]
        ax.annotate(
            f"({string.ascii_lowercase[i]})",
            xy=(0, 1.02),
            xycoords='axes fraction',
            fontsize=16,
            va='bottom',
            ha='left',
        )
        plot_lift(
            df=results,
            dataset=ds,
            metric="Multilabel (AUROC)",
            models=models,
            ax=ax,
            seed=seeds,
            do_label=i==0
        )

        ax.set_title(ds, fontsize=12, fontweight="bold")
        ax.set_ylabel("")
        if i % 3 == 0:
            if i == 0:
                domain_text = r'$\bf{No\ Overlap\ with\ HEEDB}$'
            else:
                domain_text = r'$\bf{Label\ Overlap\ with\ HEEDB}$'
            ax.set_ylabel(domain_text + '\n───────────────────────────\nMacro AUROC', fontsize=12)

        ax.set_xlabel("")
        ax.text(0.5, 0.0, 'Train Set Size',
                    ha='center', va='bottom',
                    transform=ax.transAxes)
        # ax.set_xlabel("Train Size", fontsize=12)
        ymin, ymax = ax.get_ylim()
        yticks = [y for y in np.arange(0.5, 1.1, 0.1) if y > ymin and y <= ymax]
        ax.set_yticks(yticks)
        ax.tick_params(axis='both', labelsize=12)
        # ax.set_axisbelow(True)
        ax.grid(axis="y")
    leg = axs[0, 0].get_legend()
    new_handles = [
        Line2D([0], [0], marker='o', color='w',
            markerfacecolor=h.get_facecolor()[0],
            markersize=10,
            label=t.get_text())
        for h, t in zip(leg.legend_handles, leg.get_texts())
    ]
    leg.remove()
    fig.legend(
        handles=new_handles,
        loc='lower center', # anchor point on the legend itself
        bbox_to_anchor=(0.5, -0.08), # center-bottom of the figure
        ncols=5, # one column per item = flat row
        frameon=True,
        prop={"weight": "bold", "size": 12},
        handletextpad=0.02, # space between handle and label (default 0.8)
        columnspacing=0.5,
        handleheight=1.25,
    )

    # echonext
    axs[0, 0].set_ylim([0.65, 0.83])
    # mimic
    axs[0, 1].set_ylim([0.63, 0.83])
    # zzu
    axs[0, 2].set_ylim([0.63, 0.83])
    # ptbxl
    axs[1, 0].set_ylim([0.60, 0.94])
    axs[1, 0].set_xlim([2**8, axs[1, 0].get_xlim()[1]])
    # cinc
    axs[1, 1].set_ylim([0.61, 0.91])
    # code15
    axs[1, 2].set_ylim([0.67, 1])

    fig.tight_layout(pad=0.1)
    return fig, axs

### Main Fig

In [ ]:
fig, axs = make_paper_fig([
    "ProtoSSL HEEDB",
    "ProtoSSL HEEDB (FT)",
    "SupProto HEEDB",
    "SupProto HEEDB (FT)",
    "SupProto Direct",
])

fig.savefig("figs/ecg-efficiency.png", dpi=300, bbox_inches='tight')
fig.savefig("figs/ecg-efficiency.pdf", bbox_inches='tight')

### Figs to help visualize tables

In [ ]:
fig, axs = make_paper_fig([
    "ProtoSSL HEEDB",
    "ECGFounder",
    "ST-MEM",
    "Blackbox Direct",
])

In [ ]:
plot_lift(
    df=results,
    dataset="EchoNext",
    metric="Multilabel (AUROC)",
    models=[
        "ProtoSSL HEEDB",
        "ProtoSSL HEEDB (FT)",
        "ProtoSSL HEEDB (PIA)",
        "ProtoSSL HEEDB (PIA) (FT)",
    ],
    seed=[42, 67, 70, 73, 99],
)

In [ ]:
plot_lift(
    df=results,
    dataset="EchoNext",
    metric="Multilabel (AUROC)",
    models=[
        "ProtoSSL HEEDB (83ppl)",
        "ProtoSSL HEEDB (PIA) (83ppl)",
        "ProtoSSL HEEDB (PIT)",
        "ProtoSSL HEEDB (PIP)",
    ],
    seed=[42, 67, 70, 73, 99],
)

In [ ]:
plot_lift(
    df=results,
    dataset="EchoNext",
    metric="Multilabel (AUROC)",
    models=[
        "SupProto Direct",
        "SupProto HEEDB",
        "SupProto HEEDB (FT)",
        "SupProto HEEDB (PILA)",
        "SupProto HEEDB (PILA) (FT)",
        "ProtoSSL HEEDB",
        "ProtoSSL HEEDB (FT)",
    ],
    seed=[42, 67, 70, 73, 99],
)

In [ ]:
plot_lift(
    df=results,
    dataset="EchoNext",
    metric="Multilabel (AUROC)",
    models=[
        "SupProto Direct (7ppl)",
        "SupProto HEEDB (7ppl)",
        "SupProto HEEDB (7ppl) (FT)",
        "ProtoSSL HEEDB (7ppl)",
        "ProtoSSL HEEDB (7ppl) (FT)",
    ],
    seed=[42, 67, 70, 73, 99],
)

plot_lift(
    df=results,
    dataset="EchoNext",
    metric="Multilabel (AUROC)",
    models=[
        "SupProto Direct (28ppl)",
        "SupProto HEEDB (28ppl)",
        "SupProto HEEDB (28ppl) (FT)",
        "ProtoSSL HEEDB (28ppl)",
        "ProtoSSL HEEDB (28ppl) (FT)",
    ],
    seed=[42, 67, 70, 73, 99],
)

In [ ]:
plot_lift(
    df=results,
    dataset="EchoNext",
    metric="Multilabel (AUROC)",
    models=[
        "ProtoSSL HEEDB",
        "ECGFounder (SK-OT)",
        "ECGFounder (LAP)",
        "ECGFounder (Rand)",
        "ECGFounder",
    ],
    seed=[42, 67, 70, 73, 99],
)

## Make Tables

In [ ]:
all_agg = (
    results
    .sort_values(["Model", "Dataset", "Train Size"])
    .groupby(["Dataset", "Model", "Train Size"])
    # ["Multilabel (AUROC)"].describe() # reporting std dev
    [["Multilabel (AUROC)", "AUROC 95% CI (lo)", "AUROC 95% CI (hi)"]].mean() # reporting bootstrapped CI
)
# all_agg = all_agg.reset_index()[["Dataset", "Model", "Train Size", "mean", "std"]] # reporting std dev
all_agg = all_agg.reset_index()[["Dataset", "Model", "Train Size", "Multilabel (AUROC)", "AUROC 95% CI (lo)", "AUROC 95% CI (hi)"]] # reporting bootstrapped CI

# filt_agg = all_agg[
#     all_agg["Model"].isin([
#         "SupProto Direct",
#         "SupProto HEEDB",
#         "SupProto HEEDB (FT)",
#         "ProtoSSL HEEDB",
#         "ProtoSSL HEEDB (FT)",
#     ])
# ].reset_index(drop=True)
filt_agg = all_agg

# reporting std dev
# filt_agg["val_mean"] = filt_agg["mean"].apply(lambda x: f"{x:0.3f}")
# filt_agg["val_std"] = filt_agg["std"].apply(lambda x: f"{x:.2e}")
# filt_agg["val"] = filt_agg["val_mean"] + " ± " + filt_agg["val_std"]

# report bootstrapped CI
filt_agg["val"] = (
    filt_agg["Multilabel (AUROC)"].apply(lambda x: f"{x:0.3f}")
    + " ["
    + filt_agg["AUROC 95% CI (lo)"].apply(lambda x: f"{x:0.3f}")
    + "-"
    + filt_agg["AUROC 95% CI (hi)"].apply(lambda x: f"{x:0.3f}")
    + "]"
)

pivoted = filt_agg[["Dataset", "Model", "Train Size", "val"]].pivot(columns=["Dataset", "Train Size"], index=["Model"], values="val")
pivoted = pivoted.sort_index(axis=1, level=[0, 1], ascending=[True, False])
pivoted.index.name = None

### Full Data Scale, Primary Models

In [ ]:
temp = pivoted.loc[
    ["ProtoSSL HEEDB (FT)", "SupProto HEEDB (FT)", "ProtoSSL HEEDB", "SupProto HEEDB", "SupProto Direct"],
    ["EchoNext", "MIMIC-IV-ECG", "ZZU pECG", "PTB-XL", "CinC Georgia", "CODE-15%"]
].T
temp = temp.reset_index().drop_duplicates("Dataset", keep="first").drop(columns=["Train Size"]).reset_index(drop=True)
temp = pd.concat([pd.DataFrame({"Domain": ["Out-of-Domain"]*3+["In-Domain"]*3}), temp], axis=1)
temp = temp.set_index(["Domain", "Dataset"])
temp.columns = pd.MultiIndex.from_arrays([
    ["Tuned", "Tuned", "Probed", "Probed", ""],
    ["ProtoSSL HEEDB", "SupProto HEEDB", "ProtoSSL HEEDB", "SupProto HEEDB", "SupProto Direct"],
])

print(temp.to_latex())

### All Data Scales, Primary Models

In [ ]:
print(
    pivoted.loc[
        ["ProtoSSL HEEDB (FT)", "SupProto HEEDB (FT)", "ProtoSSL HEEDB", "SupProto HEEDB", "SupProto Direct"],
        ["EchoNext", "MIMIC-IV-ECG", "ZZU pECG", "PTB-XL", "CinC Georgia", "CODE-15%"]
    ].T.to_latex()
)

### Blackbox Baselines

In [ ]:
print(
    pivoted.loc[
        ["ProtoSSL HEEDB", "ECGFounder", "ST-MEM", "Blackbox Direct"],
        ["EchoNext", "MIMIC-IV-ECG", "ZZU pECG", "PTB-XL", "CinC Georgia", "CODE-15%"]
    ].T.to_latex()
)

### Ablations

#### Num Prototypes

In [ ]:
ppl7 = pivoted.loc[
        ["ProtoSSL HEEDB (7ppl) (FT)", "SupProto HEEDB (7ppl) (FT)", "ProtoSSL HEEDB (7ppl)", "SupProto HEEDB (7ppl)", "SupProto Direct (7ppl)"],
        ["EchoNext"]
    ].T.copy().rename(columns=dict(zip(
        ["ProtoSSL HEEDB (7ppl) (FT)", "SupProto HEEDB (7ppl) (FT)", "ProtoSSL HEEDB (7ppl)", "SupProto HEEDB (7ppl)", "SupProto Direct (7ppl)"],
        ["ProtoSSL HEEDB (FT)", "SupProto HEEDB (FT)", "ProtoSSL HEEDB", "SupProto HEEDB", "SupProto Direct"]
    )))

ppl14 = pivoted.loc[
        ["ProtoSSL HEEDB (FT)", "SupProto HEEDB (FT)", "ProtoSSL HEEDB", "SupProto HEEDB", "SupProto Direct"],
        ["EchoNext"]
    ].T.copy()

ppl28 = pivoted.loc[
        ["ProtoSSL HEEDB (28ppl) (FT)", "SupProto HEEDB (28ppl) (FT)", "ProtoSSL HEEDB (28ppl)", "SupProto HEEDB (28ppl)", "SupProto Direct (28ppl)"],
        ["EchoNext"]
    ].T.copy().rename(columns=dict(zip(
        ["ProtoSSL HEEDB (28ppl) (FT)", "SupProto HEEDB (28ppl) (FT)", "ProtoSSL HEEDB (28ppl)", "SupProto HEEDB (28ppl)", "SupProto Direct (28ppl)"],
        ["ProtoSSL HEEDB (FT)", "SupProto HEEDB (FT)", "ProtoSSL HEEDB", "SupProto HEEDB", "SupProto Direct"]
    )))

ppl7["Prototypes Per Label"] = 7
ppl14["Prototypes Per Label"] = 14
ppl28["Prototypes Per Label"] = 28

ppl7.set_index("Prototypes Per Label", append=True, inplace=True)
ppl14.set_index("Prototypes Per Label", append=True, inplace=True)
ppl28.set_index("Prototypes Per Label", append=True, inplace=True)

ppl_df = pd.concat([ppl7, ppl14, ppl28]).reorder_levels([0, 2, 1]).reset_index(level=0, drop=True)
print(ppl_df.to_latex())

#### SupProto NoProj

In [ ]:
pila = pivoted.loc[
        ["ProtoSSL HEEDB (FT)", "SupProto HEEDB (PILA) (FT)", "SupProto HEEDB (FT)", "ProtoSSL HEEDB", "SupProto HEEDB (PILA)", "SupProto HEEDB"],
        ["EchoNext"]
    ].T.copy()
pila.columns = pd.MultiIndex.from_tuples([
    ("Tuned", "ProtoSSL HEEDB"),
    ("Tuned", "SupProto HEEDB (NoProj)"),
    ("Tuned", "SupProto HEEDB"),
    ("Probed", "ProtoSSL HEEDB"),
    ("Probed", "SupProto HEEDB (NoProj)"),
    ("Probed", "SupProto HEEDB"),
])
print(pila.reset_index(level=0, drop=True).to_latex())

#### LAP vs ProtoPool Assignment

In [ ]:
print(
    pivoted.loc[
        ["ProtoSSL HEEDB (FT)", "ProtoSSL HEEDB (PIA) (FT)", "ProtoSSL HEEDB", "ProtoSSL HEEDB (PIA)"],
        ["EchoNext"]
    ].T.rename(columns={
        "ProtoSSL HEEDB": "ProtoSSL HEEDB (LAP)",
        "ProtoSSL HEEDB (PIA)": "ProtoSSL HEEDB (Pool)",
    }).to_latex()
)

#### No Assignment (PIT & PIP)

In [ ]:
print(
    pivoted.loc[
        ["ProtoSSL HEEDB (83ppl)", "ProtoSSL HEEDB (PIT)", "ProtoSSL HEEDB (PIP)"],
        ["EchoNext"]
    ].T.rename(columns={
        "ProtoSSL HEEDB (83ppl)": "ProtoSSL HEEDB (LAP) (83PPL)",
    }).to_latex()
)

### Prototypes from FM

In [ ]:
print(
    pivoted.loc[
        ["ProtoSSL HEEDB", "ECGFounder (SK-OT)", "ECGFounder (LAP)", "ECGFounder (Rand)", "ECGFounder"],
        ["EchoNext"]
    ].T.to_latex()
)